# Phase 04 – Systematic Anomaly Detection Evaluation

## 1. Evaluation Event Selection

In [1]:

from google.colab import drive
from pathlib import Path
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Define dataset directory
data_root = Path("/content/drive/MyDrive/CARE_To_Compare")

# Find Wind Farm A metadata
metadata_files = [
    path for path in data_root.rglob("event_info.csv")
    if "wind farm a" in str(path.parent).lower().replace("_", " ")
]

if len(metadata_files) != 1:
    raise FileNotFoundError("Expected one Wind Farm A metadata file")

events = pd.read_csv(metadata_files[0], sep=";")

# Select candidate events for evaluation
candidate_ids = [0, 24, 26, 71, 73, 3, 17, 40]

candidates = events[
    events["event_id"].isin(candidate_ids)
][
    ["asset", "event_id", "event_label",
     "event_description", "event_start", "event_end"]
].copy()

# Display evaluation candidates
display(
    candidates.sort_values(["asset", "event_id"])
    .reset_index(drop=True)
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,asset,event_id,event_label,event_description,event_start,event_end
0,0,0,anomaly,Generator bearing failure,2023-08-06 06:10:00,2023-08-20 06:10:00
1,0,24,normal,NaN,2023-04-27 15:00:00,2023-05-11 11:20:00
2,0,26,anomaly,Hydraulic group,2023-10-12 10:20:00,2023-10-19 10:20:00
3,0,71,normal,NaN,2023-01-02 00:00:00,2023-01-16 00:00:00
4,0,73,anomaly,Hydraulic group,2023-06-10 11:40:00,2023-06-17 11:40:00
5,10,3,normal,NaN,2023-04-27 03:00:00,2023-05-18 01:10:00
6,10,17,normal,NaN,2023-11-02 15:20:00,2023-11-16 00:40:00
7,10,40,anomaly,Generator bearing failure,2022-12-26 00:00:00,2023-01-26 13:00:00


## 2. Evaluation Data Availability

In [2]:
# Check available data splits for Asset 0 events

farm_dir = metadata_files[0].parent
evaluation_event_ids = [0, 24, 26, 71, 73]

results = []

for event_id in evaluation_event_ids:
    matching_files = list(farm_dir.rglob(f"{event_id}.csv"))

    if len(matching_files) != 1:
        print(f"Event {event_id}: File not uniquely found")
        continue

    df = pd.read_csv(
        matching_files[0],
        sep=";",
        usecols=["time_stamp", "train_test"]
    )

    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    for split, group in df.groupby("train_test"):
        results.append({
            "event_id": event_id,
            "split": split,
            "observations": len(group),
            "start": group["time_stamp"].min(),
            "end": group["time_stamp"].max()
        })

availability = pd.DataFrame(results)

display(
    availability.sort_values(["event_id", "split"])
    .reset_index(drop=True)
)


,event_id,split,observations,start,end
0,0,prediction,2838,2023-08-04 06:10:00,2023-08-24 06:10:00
1,0,train,52148,2022-08-04 06:10:00,2023-08-04 06:00:00
2,24,prediction,2714,2023-04-24 15:00:00,2023-05-13 11:20:00
3,24,train,52289,2022-04-24 15:00:00,2023-04-24 14:50:00
4,26,prediction,1441,2023-10-12 10:20:00,2023-10-22 10:20:00
5,26,train,52261,2022-10-12 10:20:00,2023-10-12 10:10:00
6,71,prediction,2449,2023-01-01 00:00:00,2023-01-18 00:00:00
7,71,train,52295,2022-01-01 00:00:00,2022-12-31 23:50:00
8,73,prediction,1729,2023-06-07 11:40:00,2023-06-19 11:40:00
9,73,train,52313,2022-06-07 11:40:00,2023-06-07 11:30:00


## 3. Evaluation Window Definition

In [3]:

# Define 48-hour evaluation windows

window_hours = 48
evaluation_event_ids = [0, 24, 26, 71, 73]

window_results = []

for event_id in evaluation_event_ids:

    info = events[
        (events["asset"] == 0) &
        (events["event_id"] == event_id)
    ].iloc[0]

    event_start = pd.Timestamp(info["event_start"])

    if info["event_label"] == "anomaly":
        window_start = event_start - pd.Timedelta(hours=window_hours)
        window_end = event_start
        window_type = "pre-event"

    else:
        window_start = event_start
        window_end = event_start + pd.Timedelta(hours=window_hours)
        window_type = "normal"

    file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

    df = pd.read_csv(
        file_path,
        sep=";",
        usecols=["time_stamp", "train_test"]
    )

    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    prediction = df[df["train_test"] == "prediction"]

    selected = prediction[
        (prediction["time_stamp"] >= window_start) &
        (prediction["time_stamp"] < window_end)
    ]

    window_results.append({
        "event_id": event_id,
        "window_type": window_type,
        "observations": len(selected),
        "window_start": window_start,
        "window_end": window_end
    })

window_summary = pd.DataFrame(window_results)

display(window_summary)


,event_id,window_type,observations,window_start,window_end
0,0,pre-event,288,2023-08-04 06:10:00,2023-08-06 06:10:00
1,24,normal,288,2023-04-27 15:00:00,2023-04-29 15:00:00
2,26,pre-event,0,2023-10-10 10:20:00,2023-10-12 10:20:00
3,71,normal,288,2023-01-02 00:00:00,2023-01-04 00:00:00
4,73,pre-event,288,2023-06-08 11:40:00,2023-06-10 11:40:00


## 4. Evaluation Data Quality

In [4]:

# Apply consistent evaluation filters

model_columns = [
    "wind_speed_3_avg",
    "sensor_18_avg",
    "power_30_avg",
    "sensor_13_avg"
]

quality_results = []

for _, window in window_summary.iterrows():

    event_id = window["event_id"]

    file_path = list(
        farm_dir.rglob(f"{event_id}.csv")
    )[0]

    df = pd.read_csv(file_path, sep=";")

    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    selected = df[
        (df["train_test"] == "prediction") &
        (df["time_stamp"] >= window["window_start"]) &
        (df["time_stamp"] < window["window_end"])
    ].copy()

    for column in model_columns:
        selected[column] = pd.to_numeric(
            selected[column], errors="coerce"
        )

    # Apply Phase 03 evaluation filters
    valid = selected[
        (selected["status_type_id"] == 0) &
        (selected["sensor_18_avg"] > 0)
    ].dropna(subset=model_columns)

    quality_results.append({
        "event_id": event_id,
        "total_observations": len(selected),
        "valid_observations": len(valid),
        "excluded_observations": len(selected) - len(valid),
        "valid_rate_percent": round(
            100 * len(valid) / len(selected), 2
        ) if len(selected) > 0 else None
    })

quality_summary = pd.DataFrame(quality_results)

display(quality_summary)


,event_id,total_observations,valid_observations,excluded_observations,valid_rate_percent
0,0,288,248,40,86.11
1,24,288,288,0,100.00
2,26,0,0,0,NaN
3,71,288,288,0,100.00
4,73,288,0,288,0.00


## 5. Investigating Excluded Observations

In [5]:

# Investigate filtering for Event 73

event_id = 73

window = window_summary[
    window_summary["event_id"] == event_id
].iloc[0]

file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

df = pd.read_csv(file_path, sep=";")

df["time_stamp"] = pd.to_datetime(df["time_stamp"])

selected = df[
    (df["train_test"] == "prediction") &
    (df["time_stamp"] >= window["window_start"]) &
    (df["time_stamp"] < window["window_end"])
].copy()

# Convert model columns to numeric
for column in model_columns:
    selected[column] = pd.to_numeric(
        selected[column], errors="coerce"
    )

# Apply filters step by step
status_valid = selected[
    selected["status_type_id"] == 0
]

speed_valid = status_valid[
    status_valid["sensor_18_avg"] > 0
]

complete_valid = speed_valid.dropna(
    subset=model_columns
)

filter_summary = pd.DataFrame({
    "filter_step": [
        "All prediction observations",
        "Status == 0",
        "Generator speed > 0",
        "Complete model features"
    ],
    "remaining_observations": [
        len(selected),
        len(status_valid),
        len(speed_valid),
        len(complete_valid)
    ]
})

display(filter_summary)

# Inspect status distribution
print("Status distribution:")

display(
    selected["status_type_id"]
    .value_counts(dropna=False)
    .rename_axis("status_type_id")
    .reset_index(name="observations")
)


,filter_step,remaining_observations
0,All prediction observations,288
1,Status == 0,0
2,Generator speed > 0,0
3,Complete model features,0


Status distribution:


,status_type_id,observations
0,4,288


## 6. Event 73 – Feature Availability

In [6]:
# Check Event 73 without retrospective status filtering

complete_data = selected.dropna(subset=model_columns)

valid_speed_data = complete_data[
    complete_data["sensor_18_avg"] > 0
]

feature_quality = pd.DataFrame({
    "check": [
        "All pre-event observations",
        "Complete model features",
        "Generator speed > 0"
    ],
    "observations": [
        len(selected),
        len(complete_data),
        len(valid_speed_data)
    ]
})

display(feature_quality)

print("Missing values per feature:")
display(selected[model_columns].isna().sum())


,check,observations
0,All pre-event observations,288
1,Complete model features,288
2,Generator speed > 0,285


Missing values per feature:


,0
wind_speed_3_avg,0
sensor_18_avg,0
power_30_avg,0
sensor_13_avg,0


## 7. Consistent Evaluation Filtering

In [7]:

# Apply consistent evaluation filters to all events

evaluation_results = []

for _, window in window_summary.iterrows():

    event_id = window["event_id"]

    file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

    df = pd.read_csv(file_path, sep=";")
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    # Select prediction observations within the evaluation window
    selected = df[
        (df["train_test"] == "prediction") &
        (df["time_stamp"] >= window["window_start"]) &
        (df["time_stamp"] < window["window_end"])
    ].copy()

    # Convert required model columns to numeric
    for column in model_columns:
        selected[column] = pd.to_numeric(
            selected[column], errors="coerce"
        )

    # Apply evaluation filters without using status labels
    valid = selected[
        (selected["sensor_18_avg"] > 0)
    ].dropna(subset=model_columns)

    evaluation_results.append({
        "event_id": event_id,
        "window_type": window["window_type"],
        "available": len(selected),
        "valid": len(valid),
        "excluded": len(selected) - len(valid),
        "coverage_percent": (
            round(100 * len(valid) / len(selected), 2)
            if len(selected) > 0 else None
        )
    })

evaluation_summary = pd.DataFrame(evaluation_results)

display(evaluation_summary)


,event_id,window_type,available,valid,excluded,coverage_percent
0,0,pre-event,288,248,40,86.11
1,24,normal,288,288,0,100.00
2,26,pre-event,0,0,0,NaN
3,71,normal,288,288,0,100.00
4,73,pre-event,288,285,3,98.96


## 8. Baseline Model Evaluation

In [8]:
import joblib

# Load the trained Phase 03 baseline
artifact_path = (
    "/content/drive/MyDrive/Industrial_AI_Artifacts/"
    "baseline_phase03_asset0.joblib"
)

baseline = joblib.load(artifact_path)

model = baseline["model"]
features = list(baseline["feature_cols"])
target = baseline["target_col"]
threshold = baseline["threshold"]

# Apply the unchanged model to evaluation windows
results = []
scored_events = {}

for _, window in window_summary.iterrows():

    event_id = window["event_id"]
    file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

    df = pd.read_csv(file_path, sep=";")
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    selected = df[
        (df["train_test"] == "prediction") &
        (df["time_stamp"] >= window["window_start"]) &
        (df["time_stamp"] < window["window_end"])
    ].copy()

    for col in features + [target]:
        selected[col] = pd.to_numeric(
            selected[col], errors="coerce"
        )

    valid = selected[
        selected["sensor_18_avg"] > 0
    ].dropna(subset=features + [target]).copy()

    if valid.empty:
        print(f"Event {event_id}: No valid observations")
        continue

    valid = valid.sort_values("time_stamp")

    # Generate predictions without retraining
    valid["predicted"] = model.predict(valid[features])

    valid["residual"] = (
        valid[target] - valid["predicted"]
    )

    valid["alert"] = valid["residual"] > threshold

    scored_events[event_id] = valid

    results.append({
        "event_id": event_id,
        "valid_observations": len(valid),
        "mean_residual": round(valid["residual"].mean(), 2),
        "max_residual": round(valid["residual"].max(), 2),
        "alerts": int(valid["alert"].sum()),
        "alert_rate_percent": round(
            100 * valid["alert"].mean(), 2
        )
    })

evaluation_metrics = pd.DataFrame(results)

display(evaluation_metrics)


Event 26: No valid observations


,event_id,valid_observations,mean_residual,max_residual,alerts,alert_rate_percent
0,0,248,10.28,23.76,15,6.05
1,24,288,-4.96,17.55,0,0.00
2,71,288,-9.22,4.69,0,0.00
3,73,285,10.53,25.74,30,10.53


## 9. Alert Grouping and Lead Time

In [9]:

# Group consecutive alerts and calculate observed lead time

alert_groups = []

for event_id, scored in sorted(scored_events.items()):

    alert_times = (
        scored.loc[scored["alert"], "time_stamp"]
        .sort_values()
        .reset_index(drop=True)
    )

    if alert_times.empty:
        continue

    # Start a new group when alerts are not 10 minutes apart
    group_ids = (
        alert_times.diff()
        .ne(pd.Timedelta(minutes=10))
        .cumsum()
    )

    info = events[
        (events["asset"] == 0) &
        (events["event_id"] == event_id)
    ].iloc[0]

    event_start = pd.Timestamp(info["event_start"])

    for group_number, (_, times) in enumerate(
        alert_times.groupby(group_ids), start=1
    ):

        first_alert = times.iloc[0]
        last_alert = times.iloc[-1]

        lead_hours = (
            round(
                (event_start - first_alert).total_seconds() / 3600,
                2
            )
            if info["event_label"] == "anomaly"
            else None
        )

        alert_groups.append({
            "event_id": event_id,
            "group": group_number,
            "first_alert": first_alert,
            "last_alert": last_alert,
            "alerts": len(times),
            "lead_hours": lead_hours
        })

alert_group_summary = pd.DataFrame(alert_groups)

display(alert_group_summary)


,event_id,group,first_alert,last_alert,alerts,lead_hours
0,0,1,2023-08-04 17:40:00,2023-08-04 18:00:00,3,36.50
1,0,2,2023-08-04 18:30:00,2023-08-04 18:30:00,1,35.67
2,0,3,2023-08-04 19:30:00,2023-08-04 20:20:00,6,34.67
3,0,4,2023-08-04 20:50:00,2023-08-04 21:00:00,2,33.33
4,0,5,2023-08-04 21:20:00,2023-08-04 21:30:00,2,32.83
5,0,6,2023-08-05 20:40:00,2023-08-05 20:40:00,1,9.50
6,73,1,2023-06-08 15:30:00,2023-06-08 15:30:00,1,44.17
7,73,2,2023-06-08 16:00:00,2023-06-08 16:00:00,1,43.67
8,73,3,2023-06-08 18:10:00,2023-06-08 18:20:00,2,41.50
9,73,4,2023-06-08 19:20:00,2023-06-08 20:00:00,5,40.33


## 10. Event-Based Evaluation Summary

In [10]:

# Create an event-based evaluation summary

event_results = []

for _, row in evaluation_summary.iterrows():

    event_id = int(row["event_id"])

    info = events[
        (events["asset"] == 0) &
        (events["event_id"] == event_id)
    ].iloc[0]

    label = info["event_label"]
    valid_count = int(row["valid"])

    if valid_count == 0:
        alerts = None
        groups = None
        flagged = None
        lead_hours = None

    else:
        scored = scored_events[event_id]
        alerts = int(scored["alert"].sum())

        groups = int(
            (alert_group_summary["event_id"] == event_id).sum()
        )

        flagged = alerts > 0
        lead_hours = None

        if label == "anomaly" and flagged:
            first_alert = scored.loc[
                scored["alert"], "time_stamp"
            ].min()

            event_start = pd.Timestamp(info["event_start"])

            lead_hours = round(
                (event_start - first_alert).total_seconds() / 3600,
                2
            )

    event_results.append({
        "event_id": event_id,
        "label": label,
        "valid_observations": valid_count,
        "alerts": alerts,
        "alarm_groups": groups,
        "event_flagged": flagged,
        "first_alert_lead_hours": lead_hours
    })

event_summary = pd.DataFrame(event_results)

display(event_summary)


,event_id,label,valid_observations,alerts,alarm_groups,event_flagged,first_alert_lead_hours
0,0,anomaly,248,15.0,6.0,True,36.50
1,24,normal,288,0.0,0.0,False,NaN
2,26,anomaly,0,NaN,NaN,None,NaN
3,71,normal,288,0.0,0.0,False,NaN
4,73,anomaly,285,30.0,13.0,True,44.17


## 11. Event-Based Detection Metrics

In [11]:

# Calculate observed event-based evaluation metrics

evaluable = event_summary[
    event_summary["valid_observations"] > 0
].copy()

anomaly_events = evaluable[
    evaluable["label"] == "anomaly"
]

normal_events = evaluable[
    evaluable["label"] == "normal"
]

detected_anomalies = int(
    anomaly_events["event_flagged"].eq(True).sum()
)

flagged_normals = int(
    normal_events["event_flagged"].eq(True).sum()
)

metrics_summary = pd.DataFrame({
    "metric": [
        "Observed Event Detection Rate",
        "Observed Normal Window Alarm Rate"
    ],
    "numerator": [
        detected_anomalies,
        flagged_normals
    ],
    "denominator": [
        len(anomaly_events),
        len(normal_events)
    ]
})

metrics_summary["rate_percent"] = (
    100 * metrics_summary["numerator"]
    / metrics_summary["denominator"].replace(0, float("nan"))
).round(2)

display(metrics_summary)

# Document excluded anomaly events
excluded_anomalies = event_summary[
    (event_summary["label"] == "anomaly") &
    (event_summary["valid_observations"] == 0)
]

print(
    "Excluded anomaly events:",
    excluded_anomalies["event_id"].tolist()
)


,metric,numerator,denominator,rate_percent
0,Observed Event Detection Rate,2,2,100.0
1,Observed Normal Window Alarm Rate,0,2,0.0


Excluded anomaly events: [26]


## 12. Extended Normal Operation Evaluation

In [12]:

# Evaluate the full documented normal period of Event 24

event_id = 24

info = events[
    (events["asset"] == 0) &
    (events["event_id"] == event_id)
].iloc[0]

normal_start = pd.Timestamp(info["event_start"])
normal_end = pd.Timestamp(info["event_end"])

file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

df = pd.read_csv(file_path, sep=";")
df["time_stamp"] = pd.to_datetime(df["time_stamp"])

# Use only prediction data within the documented normal period
selected = df[
    (df["train_test"] == "prediction") &
    (df["time_stamp"] >= normal_start) &
    (df["time_stamp"] < normal_end)
].copy()

for col in features + [target]:
    selected[col] = pd.to_numeric(selected[col], errors="coerce")

# Apply the unchanged evaluation filters
valid = selected[
    selected["sensor_18_avg"] > 0
].dropna(subset=features + [target]).copy()

valid = valid.sort_values("time_stamp")

# Apply the existing baseline
valid["predicted"] = model.predict(valid[features])
valid["residual"] = valid[target] - valid["predicted"]
valid["alert"] = valid["residual"] > threshold

normal_summary = pd.DataFrame([{
    "event_id": event_id,
    "available_observations": len(selected),
    "valid_observations": len(valid),
    "alerts": int(valid["alert"].sum()),
    "alert_rate_percent": round(
        100 * valid["alert"].mean(), 2
    ) if len(valid) > 0 else None
}])

display(normal_summary)

print("Alert timestamps:")
display(
    valid.loc[
        valid["alert"],
        ["time_stamp", "residual"]
    ]
)


,event_id,available_observations,valid_observations,alerts,alert_rate_percent
0,24,1994,1968,3,0.15


Alert timestamps:


,time_stamp,residual
53506,2023-05-03 02:00:00,18.908953
53622,2023-05-03 21:20:00,19.914431
53623,2023-05-03 21:30:00,20.168242


## 13. Normal Operation Alarm Frequency

In [13]:

# Group alerts from the extended Event 24 evaluation

alert_times = (
    valid.loc[valid["alert"], "time_stamp"]
    .sort_values()
    .reset_index(drop=True)
)

group_ids = (
    alert_times.diff()
    .ne(pd.Timedelta(minutes=10))
    .cumsum()
)

alert_data = pd.DataFrame({
    "time_stamp": alert_times,
    "group_id": group_ids
})

normal_alarm_groups = (
    alert_data.groupby("group_id")
    .agg(
        first_alert=("time_stamp", "min"),
        last_alert=("time_stamp", "max"),
        alerts=("time_stamp", "size")
    )
    .reset_index(drop=True)
)

# Calculate frequency per 24 hours of valid observations
sampling_minutes = 10

valid_hours = len(valid) * sampling_minutes / 60

alarms_per_24h = (
    len(normal_alarm_groups) / valid_hours * 24
)

display(normal_alarm_groups)

print("Valid observation hours:", round(valid_hours, 2))
print("Alarm groups:", len(normal_alarm_groups))
print("Alarm groups per 24 valid hours:", round(alarms_per_24h, 3))


,first_alert,last_alert,alerts
0,2023-05-03 02:00:00,2023-05-03 02:00:00,1
1,2023-05-03 21:20:00,2023-05-03 21:30:00,2


Valid observation hours: 328.0
Alarm groups: 2
Alarm groups per 24 valid hours: 0.146


## 14. Extended Normal Evaluation – Event 71

In [14]:

# Evaluate the full documented normal period of Event 71

event_id = 71

info = events[
    (events["asset"] == 0) &
    (events["event_id"] == event_id)
].iloc[0]

normal_start = pd.Timestamp(info["event_start"])
normal_end = pd.Timestamp(info["event_end"])

file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

df = pd.read_csv(file_path, sep=";")
df["time_stamp"] = pd.to_datetime(df["time_stamp"])

# Select prediction data within the normal event
selected = df[
    (df["train_test"] == "prediction") &
    (df["time_stamp"] >= normal_start) &
    (df["time_stamp"] < normal_end)
].copy()

for col in features + [target]:
    selected[col] = pd.to_numeric(
        selected[col], errors="coerce"
    )

# Apply unchanged evaluation filters
event71_valid = selected[
    selected["sensor_18_avg"] > 0
].dropna(subset=features + [target]).copy()

event71_valid = event71_valid.sort_values("time_stamp")

# Apply the existing baseline
event71_valid["predicted"] = model.predict(
    event71_valid[features]
)

event71_valid["residual"] = (
    event71_valid[target] - event71_valid["predicted"]
)

event71_valid["alert"] = (
    event71_valid["residual"] > threshold
)

summary71 = pd.DataFrame([{
    "event_id": event_id,
    "available": len(selected),
    "valid": len(event71_valid),
    "alerts": int(event71_valid["alert"].sum()),
    "alert_rate_percent": round(
        100 * event71_valid["alert"].mean(), 2
    )
}])

display(summary71)

print("First 10 alert timestamps:")

display(
    event71_valid.loc[
        event71_valid["alert"],
        ["time_stamp", "residual"]
    ].head(10)
)


,event_id,available,valid,alerts,alert_rate_percent
0,71,2016,1858,0,0.0


First 10 alert timestamps:


,time_stamp,residual


## 15. Combined Normal Operation Evaluation

In [15]:

# Compare alarm frequency across both normal events

normal_datasets = {
    24: valid,
    71: event71_valid
}

comparison_results = []

for event_id, data in normal_datasets.items():

    # Check timestamp uniqueness
    assert data["time_stamp"].is_unique, (
        f"Duplicate timestamps in Event {event_id}"
    )

    alert_times = (
        data.loc[data["alert"], "time_stamp"]
        .sort_values()
        .reset_index(drop=True)
    )

    # Count consecutive alert groups
    if alert_times.empty:
        group_count = 0
    else:
        group_count = int(
            alert_times.diff()
            .ne(pd.Timedelta(minutes=10))
            .sum()
        )

    valid_hours = len(data) * 10 / 60

    comparison_results.append({
        "event_id": event_id,
        "valid_observations": len(data),
        "alerts": len(alert_times),
        "alarm_groups": group_count,
        "valid_hours": round(valid_hours, 2),
        "groups_per_24h": round(
            group_count / valid_hours * 24, 3
        )
    })

normal_comparison = pd.DataFrame(comparison_results)

display(normal_comparison)

# Calculate combined metrics
total_groups = normal_comparison["alarm_groups"].sum()
total_hours = normal_comparison["valid_observations"].sum() / 6

print("Combined alarm groups:", total_groups)
print("Combined valid hours:", round(total_hours, 2))
print(
    "Combined groups per 24 valid hours:",
    round(total_groups / total_hours * 24, 3)
)


,event_id,valid_observations,alerts,alarm_groups,valid_hours,groups_per_24h
0,24,1968,3,2,328.00,0.146
1,71,1858,0,0,309.67,0.000


Combined alarm groups: 2
Combined valid hours: 637.67
Combined groups per 24 valid hours: 0.075


## 16. Cross-Asset Evaluation – Data Availability

In [16]:
# Check evaluation window availability for Asset 10

asset_id = 10
candidate_event_ids = [3, 17, 40]

asset10_results = []

for event_id in candidate_event_ids:

    info = events[
        (events["asset"] == asset_id) &
        (events["event_id"] == event_id)
    ].iloc[0]

    event_start = pd.Timestamp(info["event_start"])
    label = info["event_label"]

    # Define evaluation window
    if label == "anomaly":
        window_start = event_start - pd.Timedelta(hours=48)
        window_end = event_start
    else:
        window_start = event_start
        window_end = event_start + pd.Timedelta(hours=48)

    matching_files = list(farm_dir.rglob(f"{event_id}.csv"))

    if len(matching_files) != 1:
        raise FileNotFoundError(
            f"Expected one CSV file for Event {event_id}"
        )

    df = pd.read_csv(
        matching_files[0],
        sep=";",
        usecols=["time_stamp", "train_test"]
    )

    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    prediction = df[df["train_test"] == "prediction"]

    selected = prediction[
        (prediction["time_stamp"] >= window_start) &
        (prediction["time_stamp"] < window_end)
    ]

    asset10_results.append({
        "asset": asset_id,
        "event_id": event_id,
        "label": label,
        "window_start": window_start,
        "window_end": window_end,
        "available_observations": len(selected)
    })

asset10_availability = pd.DataFrame(asset10_results)

display(asset10_availability)


,asset,event_id,label,window_start,window_end,available_observations
0,10,3,normal,2023-04-27 03:00:00,2023-04-29 03:00:00,288
1,10,17,normal,2023-11-02 15:20:00,2023-11-04 15:20:00,288
2,10,40,anomaly,2022-12-24 00:00:00,2022-12-26 00:00:00,144


## 17. Cross-Asset Evaluation – Event 40 Time Coverage

In [17]:
# Investigate timestamp coverage for Event 40

event_id = 40

window = asset10_availability[
    asset10_availability["event_id"] == event_id
].iloc[0]

file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

df = pd.read_csv(
    file_path,
    sep=";",
    usecols=["time_stamp", "train_test"]
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"])

selected = df[
    (df["train_test"] == "prediction") &
    (df["time_stamp"] >= window["window_start"]) &
    (df["time_stamp"] < window["window_end"])
].sort_values("time_stamp")

timestamps = selected["time_stamp"]

# Expected 10-minute timestamps
expected = pd.date_range(
    start=window["window_start"],
    end=window["window_end"],
    freq="10min",
    inclusive="left"
)

print("Event:", event_id)
print("Available observations:", len(selected))
print("Unique timestamps:", timestamps.nunique())
print("First observation:", timestamps.min())
print("Last observation:", timestamps.max())

print(
    "Missing expected timestamps:",
    (~expected.isin(timestamps)).sum()
)

print(
    "Largest observed time gap:",
    timestamps.diff().max()
)


Event: 40
Available observations: 144
Unique timestamps: 144
First observation: 2022-12-25 00:00:00
Last observation: 2022-12-25 23:50:00
Missing expected timestamps: 144
Largest observed time gap: 0 days 00:10:00


## 18. Cross-Asset Evaluation – Data Quality

In [18]:

# Check model feature availability for Asset 10

asset10_quality_results = []

for _, window in asset10_availability.iterrows():

    event_id = int(window["event_id"])

    file_path = list(
        farm_dir.rglob(f"{event_id}.csv")
    )[0]

    df = pd.read_csv(file_path, sep=";")

    df["time_stamp"] = pd.to_datetime(
        df["time_stamp"]
    )

    # Select the defined prediction window
    selected = df[
        (df["train_test"] == "prediction") &
        (df["time_stamp"] >= window["window_start"]) &
        (df["time_stamp"] < window["window_end"])
    ].copy()

    # Convert model columns to numeric
    for col in features + [target]:
        selected[col] = pd.to_numeric(
            selected[col], errors="coerce"
        )

    # Check complete model values
    complete = selected.dropna(
        subset=features + [target]
    )

    # Apply the same technical evaluation filter
    valid = complete[
        complete["sensor_18_avg"] > 0
    ]

    asset10_quality_results.append({
        "event_id": event_id,
        "label": window["label"],
        "available": len(selected),
        "complete_features": len(complete),
        "valid": len(valid),
        "excluded": len(selected) - len(valid),
        "coverage_percent": (
            round(100 * len(valid) / len(selected), 2)
            if len(selected) > 0 else None
        )
    })

asset10_quality = pd.DataFrame(
    asset10_quality_results
)

display(asset10_quality)


,event_id,label,available,complete_features,valid,excluded,coverage_percent
0,3,normal,288,288,288,0,100.0
1,17,normal,288,288,288,0,100.0
2,40,anomaly,144,144,144,0,100.0


## 19. Cross-Asset Model Evaluation

In [19]:

# Evaluate the existing Asset 0 baseline on Asset 10

asset10_scored = []
transfer_results = []

for _, window in asset10_availability.iterrows():

    event_id = int(window["event_id"])

    file_path = list(farm_dir.rglob(f"{event_id}.csv"))[0]

    df = pd.read_csv(file_path, sep=";")
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])

    # Select the predefined evaluation window
    selected = df[
        (df["train_test"] == "prediction") &
        (df["time_stamp"] >= window["window_start"]) &
        (df["time_stamp"] < window["window_end"])
    ].copy()

    for col in features + [target]:
        selected[col] = pd.to_numeric(
            selected[col], errors="coerce"
        )

    # Apply unchanged technical evaluation filters
    valid = selected[
        selected["sensor_18_avg"] > 0
    ].dropna(subset=features + [target]).copy()

    valid = valid.sort_values("time_stamp")

    if valid.empty:
        continue

    # Use the original Asset 0 model
    valid["predicted"] = model.predict(valid[features])

    valid["residual"] = (
        valid[target] - valid["predicted"]
    )

    valid["alert"] = valid["residual"] > threshold

    valid["event_id"] = event_id
    asset10_scored.append(valid)

    transfer_results.append({
        "event_id": event_id,
        "label": window["label"],
        "valid": len(valid),
        "mean_residual": round(valid["residual"].mean(), 2),
        "mae": round(valid["residual"].abs().mean(), 2),
        "max_residual": round(valid["residual"].max(), 2),
        "alerts": int(valid["alert"].sum()),
        "alert_rate_percent": round(
            100 * valid["alert"].mean(), 2
        )
    })

asset10_metrics = pd.DataFrame(transfer_results)

display(asset10_metrics)


,event_id,label,valid,mean_residual,mae,max_residual,alerts,alert_rate_percent
0,3,normal,288,-8.80,9.18,12.16,0,0.0
1,17,normal,288,-6.50,7.12,7.15,0,0.0
2,40,anomaly,144,-9.86,9.89,1.82,0,0.0


## 20. Cross-Asset Residual Bias Analysis

In [20]:
# Investigate prediction bias across Asset 10 events

bias_results = []

for data in asset10_scored:

    event_id = int(data["event_id"].iloc[0])

    bias_results.append({
        "event_id": event_id,
        "actual_mean": round(data[target].mean(), 2),
        "predicted_mean": round(data["predicted"].mean(), 2),
        "median_residual": round(data["residual"].median(), 2),
        "residual_p05": round(data["residual"].quantile(0.05), 2),
        "residual_p95": round(data["residual"].quantile(0.95), 2),
        "positive_residuals": int(
            (data["residual"] > 0).sum()
        )
    })

bias_summary = pd.DataFrame(bias_results)

display(bias_summary)


,event_id,actual_mean,predicted_mean,median_residual,residual_p05,residual_p95,positive_residuals
0,3,38.83,47.63,-8.42,-19.07,0.62,18
1,17,36.11,42.60,-6.86,-15.29,2.61,39
2,40,47.37,57.23,-9.80,-17.79,-3.19,2


## 21. Cross-Asset Operating Conditions Comparison

In [21]:

# Compare normal operating conditions across assets

asset10_by_event = {
    int(data["event_id"].iloc[0]): data
    for data in asset10_scored
}

normal_cases = [
    (0, 24, scored_events[24]),
    (0, 71, scored_events[71]),
    (10, 3, asset10_by_event[3]),
    (10, 17, asset10_by_event[17])
]

comparison_rows = []

for asset_id, event_id, data in normal_cases:

    comparison_rows.append({
        "asset": asset_id,
        "event_id": event_id,
        "wind_speed_mean": round(
            data["wind_speed_3_avg"].mean(), 2
        ),
        "generator_speed_mean": round(
            data["sensor_18_avg"].mean(), 2
        ),
        "power_mean": round(
            data["power_30_avg"].mean(), 2
        ),
        "actual_temp_mean": round(
            data[target].mean(), 2
        ),
        "predicted_temp_mean": round(
            data["predicted"].mean(), 2
        ),
        "mean_residual": round(
            data["residual"].mean(), 2
        )
    })

operating_comparison = pd.DataFrame(comparison_rows)

display(operating_comparison)


,asset,event_id,wind_speed_mean,generator_speed_mean,power_mean,actual_temp_mean,predicted_temp_mean,mean_residual
0,0,24,7.62,1346.13,0.46,46.17,51.13,-4.96
1,0,71,7.47,1475.53,0.41,41.85,51.07,-9.22
2,10,3,6.65,1208.66,0.36,38.83,47.63,-8.80
3,10,17,5.18,1138.24,0.16,36.11,42.60,-6.50


## 22. Matched 24-Hour Generator Bearing Evaluation

In [22]:

# Compare both bearing failure events over 24 hours

failure_cases = [
    (0, 0, scored_events[0]),
    (10, 40, asset10_by_event[40])
]

matched_results = []

for asset_id, event_id, data in failure_cases:

    info = events[
        (events["asset"] == asset_id) &
        (events["event_id"] == event_id)
    ].iloc[0]

    event_start = pd.Timestamp(info["event_start"])
    window_start = event_start - pd.Timedelta(hours=24)

    # Select the same 24-hour pre-event period
    matched = data[
        (data["time_stamp"] >= window_start) &
        (data["time_stamp"] < event_start)
    ].copy()

    alerts = matched[matched["alert"]]

    lead_hours = None

    if not alerts.empty:
        first_alert = alerts["time_stamp"].min()
        lead_hours = round(
            (event_start - first_alert).total_seconds() / 3600,
            2
        )

    matched_results.append({
        "asset": asset_id,
        "event_id": event_id,
        "valid_observations": len(matched),
        "alerts": len(alerts),
        "mean_residual": round(
            matched["residual"].mean(), 2
        ),
        "max_residual": round(
            matched["residual"].max(), 2
        ),
        "first_alert_lead_hours": lead_hours
    })

matched_comparison = pd.DataFrame(matched_results)

display(matched_comparison)


,asset,event_id,valid_observations,alerts,mean_residual,max_residual,first_alert_lead_hours
0,0,0,118,1,10.18,19.20,9.5
1,10,40,144,0,-9.86,1.82,NaN


## 23. Matched Window Data Coverage Analysis

In [23]:

# Investigate excluded observations in Event 0

event_id = 0

info = events[
    (events["asset"] == 0) &
    (events["event_id"] == event_id)
].iloc[0]

event_start = pd.Timestamp(info["event_start"])
window_start = event_start - pd.Timedelta(hours=24)

file_path = list(farm_dir.rglob("0.csv"))[0]

df = pd.read_csv(file_path, sep=";")
df["time_stamp"] = pd.to_datetime(df["time_stamp"])

selected = df[
    (df["train_test"] == "prediction") &
    (df["time_stamp"] >= window_start) &
    (df["time_stamp"] < event_start)
].copy()

# Check model feature completeness
for col in features + [target]:
    selected[col] = pd.to_numeric(
        selected[col], errors="coerce"
    )

complete = selected.dropna(
    subset=features + [target]
)

valid = complete[
    complete["sensor_18_avg"] > 0
].sort_values("time_stamp")

coverage_summary = pd.DataFrame({
    "check": [
        "Available observations",
        "Excluded: missing model values",
        "Excluded: generator speed <= 0",
        "Valid observations"
    ],
    "observations": [
        len(selected),
        len(selected) - len(complete),
        len(complete) - len(valid),
        len(valid)
    ]
})

display(coverage_summary)

print("First valid timestamp:", valid["time_stamp"].min())
print("Last valid timestamp:", valid["time_stamp"].max())
print(
    "Largest gap between valid observations:",
    valid["time_stamp"].diff().max()
)


,check,observations
0,Available observations,144
1,Excluded: missing model values,0
2,Excluded: generator speed <= 0,26
3,Valid observations,118


First valid timestamp: 2023-08-05 06:10:00
Last valid timestamp: 2023-08-06 06:00:00
Largest gap between valid observations: 0 days 01:10:00


## 24. Excluded Operating Period Analysis

In [24]:
# Identify excluded operating periods in Event 0

invalid = selected[
    selected["sensor_18_avg"] <= 0
].sort_values("time_stamp").copy()

# Group consecutive excluded observations
invalid["group_id"] = (
    invalid["time_stamp"]
    .diff()
    .ne(pd.Timedelta(minutes=10))
    .cumsum()
)

excluded_groups = (
    invalid.groupby("group_id")
    .agg(
        first_excluded=("time_stamp", "min"),
        last_excluded=("time_stamp", "max"),
        observations=("time_stamp", "size")
    )
    .reset_index(drop=True)
)

# Duration represented by excluded measurements
excluded_groups["duration_minutes"] = (
    excluded_groups["observations"] * 10
)

display(excluded_groups)

print("Excluded observations:", len(invalid))
print("Excluded groups:", len(excluded_groups))
print(
    "Zero-speed observations:",
    int((invalid["sensor_18_avg"] == 0).sum())
)
print(
    "Negative-speed observations:",
    int((invalid["sensor_18_avg"] < 0).sum())
)


,first_excluded,last_excluded,observations,duration_minutes
0,2023-08-05 09:10:00,2023-08-05 09:20:00,2,20
1,2023-08-05 09:40:00,2023-08-05 10:30:00,6,60
2,2023-08-05 22:10:00,2023-08-05 23:00:00,6,60
3,2023-08-06 02:00:00,2023-08-06 02:00:00,1,10
4,2023-08-06 03:10:00,2023-08-06 03:10:00,1,10
5,2023-08-06 03:50:00,2023-08-06 04:10:00,3,30
6,2023-08-06 04:40:00,2023-08-06 05:10:00,4,40
7,2023-08-06 05:30:00,2023-08-06 05:50:00,3,30


Excluded observations: 26
Excluded groups: 8
Zero-speed observations: 26
Negative-speed observations: 0


## 25. Consolidated Evaluation Results

In [25]:

# Consolidate evaluation results across both assets

# Asset 0 evaluation
asset0_summary = event_summary[
    ["event_id", "label", "valid_observations",
     "alerts", "event_flagged"]
].copy()

asset0_summary.insert(0, "asset", 0)

asset0_summary["available_observations"] = (
    asset0_summary["event_id"].map(
        evaluation_summary.set_index("event_id")["available"]
    )
)

# Asset 10 evaluation
asset10_summary = asset10_metrics[
    ["event_id", "label", "valid", "alerts"]
].rename(
    columns={"valid": "valid_observations"}
).copy()

asset10_summary.insert(0, "asset", 10)

asset10_summary["event_flagged"] = (
    asset10_summary["alerts"] > 0
)

asset10_summary["available_observations"] = (
    asset10_summary["event_id"].map(
        asset10_availability.set_index("event_id")[
            "available_observations"
        ]
    )
)

# Combine event summaries
final_evaluation = pd.concat(
    [asset0_summary, asset10_summary],
    ignore_index=True
)

final_evaluation = final_evaluation.sort_values(
    ["asset", "event_id"]
).reset_index(drop=True)

display(final_evaluation)


,asset,event_id,label,valid_observations,alerts,event_flagged,available_observations
0,0,0,anomaly,248,15.0,True,288
1,0,24,normal,288,0.0,False,288
2,0,26,anomaly,0,NaN,None,0
3,0,71,normal,288,0.0,False,288
4,0,73,anomaly,285,30.0,True,288
5,10,3,normal,288,0.0,False,288
6,10,17,normal,288,0.0,False,288
7,10,40,anomaly,144,0.0,False,144


## 26. Evaluation Protocol Documentation

In [26]:
# Document evaluation conditions for reproducibility

final_evaluation = final_evaluation.copy()

# Identify evaluation scope
final_evaluation["evaluation_scope"] = (
    final_evaluation["asset"].map({
        0: "same_asset",
        10: "cross_asset"
    })
)

# Record predefined window duration
final_evaluation["window_hours"] = 48

final_evaluation.loc[
    (final_evaluation["asset"] == 10) &
    (final_evaluation["event_id"] == 40),
    "window_hours"
] = 24

# Mark events without evaluation data
final_evaluation["evaluation_status"] = "evaluable"

final_evaluation.loc[
    final_evaluation["available_observations"] == 0,
    "evaluation_status"
] = "not_evaluable"

# Calculate valid observation coverage
final_evaluation["coverage_percent"] = (
    100 *
    final_evaluation["valid_observations"] /
    final_evaluation["available_observations"].replace(
        0, float("nan")
    )
).round(2)

display(final_evaluation)


,asset,event_id,label,valid_observations,alerts,event_flagged,available_observations,evaluation_scope,window_hours,evaluation_status,coverage_percent
0,0,0,anomaly,248,15.0,True,288,same_asset,48,evaluable,86.11
1,0,24,normal,288,0.0,False,288,same_asset,48,evaluable,100.00
2,0,26,anomaly,0,NaN,None,0,same_asset,48,not_evaluable,NaN
3,0,71,normal,288,0.0,False,288,same_asset,48,evaluable,100.00
4,0,73,anomaly,285,30.0,True,288,same_asset,48,evaluable,98.96
5,10,3,normal,288,0.0,False,288,cross_asset,48,evaluable,100.00
6,10,17,normal,288,0.0,False,288,cross_asset,48,evaluable,100.00
7,10,40,anomaly,144,0.0,False,144,cross_asset,24,evaluable,100.00


## 27. Evaluation Results Export

In [27]:

from pathlib import Path

# Prepare reproducible evaluation results
protocol_table = final_evaluation.rename(
    columns={
        "coverage_percent": "filter_pass_rate_percent"
    }
).drop(columns=["window_hours"]).copy()

# All events were initially assigned a 48-hour window
protocol_table["planned_window_hours"] = 48

# Convert observation counts to 10-minute equivalents
protocol_table["available_hours_equivalent"] = (
    protocol_table["available_observations"] / 6
).round(2)

# Coverage relative to the planned window
protocol_table["prediction_window_coverage_percent"] = (
    100 * protocol_table["available_observations"]
    / (48 * 6)
).round(2)

# Save evaluation results
output_dir = Path(
    "/content/drive/MyDrive/Industrial_AI_Artifacts/phase04"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "phase04_event_window_summary.csv"

protocol_table.to_csv(output_file, index=False)

display(protocol_table[[
    "asset",
    "event_id",
    "evaluation_status",
    "planned_window_hours",
    "available_hours_equivalent",
    "prediction_window_coverage_percent",
    "filter_pass_rate_percent"
]])

print("Evaluation results saved:", output_file)


,asset,event_id,evaluation_status,planned_window_hours,available_hours_equivalent,prediction_window_coverage_percent,filter_pass_rate_percent
0,0,0,evaluable,48,48.0,100.0,86.11
1,0,24,evaluable,48,48.0,100.0,100.00
2,0,26,not_evaluable,48,0.0,0.0,NaN
3,0,71,evaluable,48,48.0,100.0,100.00
4,0,73,evaluable,48,48.0,100.0,98.96
5,10,3,evaluable,48,48.0,100.0,100.00
6,10,17,evaluable,48,48.0,100.0,100.00
7,10,40,evaluable,48,24.0,50.0,100.00


Evaluation results saved: /content/drive/MyDrive/Industrial_AI_Artifacts/phase04/phase04_event_window_summary.csv


## 28. Scientific Evaluation Summary

In [28]:

from pathlib import Path

# Prepare Phase 04 scientific evaluation report

report_text = """# Phase 04 – Anomaly Detection Evaluation

## Objective

Evaluate the existing residual-based Linear Regression
baseline on multiple wind turbine events without retraining
the model or modifying the anomaly threshold.

## Methodology

- Dataset: CARE to Compare, Wind Farm A
- Training asset: Asset 0
- Evaluation assets: Asset 0 and Asset 10
- Model: Linear Regression
- Anomaly score: Actual temperature - Predicted temperature
- Alert threshold: Original Phase 03 training-based threshold
- Evaluation filtering: Generator speed > 0 and complete model values
- No retrospective status label filtering
- Nominal evaluation window: 48 hours

## Key Findings

### Same-Asset Evaluation

- Two evaluable anomaly events produced alerts.
- Event 0: 15 alerts; first observed lead time 36.50 hours.
- Event 73: 30 alerts; first observed lead time 44.17 hours.
- Event 26: Not evaluable due to unavailable pre-event prediction data.
- Both selected 48-hour normal windows produced zero alerts.

### Extended Normal Operation

- Event 24: 3 alerts in 2 alarm groups.
- Event 71: 0 alerts.
- Combined alarm frequency: 0.075 groups per 24 valid
  observation hours.

### Cross-Asset Evaluation

- Normal Events 3 and 17: No alerts.
- Generator bearing failure Event 40: No alerts.
- Event 40 provides only 24 hours of pre-event data.
- Cross-asset early-warning capability was not demonstrated.

## Limitations

- Small and non-representative event sample.
- Exploratory event selection and evaluation.
- Unequal pre-event data availability.
- Possible differences in operating conditions.
- Temporal gaps caused by excluded operating observations.
- Potential prediction bias and limited model transferability.
- Retrospective event annotations do not establish physical
  fault onset times.
- No independently calibrated anomaly threshold.
- Observed detection rates do not establish generalization
  performance or reliable early-warning capability.

## Conclusion

The baseline identified elevated temperature residuals before
two documented events on the training asset.

However, it failed to flag the evaluated bearing failure
on Asset 10.

The results support further systematic evaluation and
model development, but do not demonstrate reliable
predictive maintenance performance.
"""

report_file = output_dir / "phase04_evaluation_summary.md"

report_file.write_text(report_text, encoding="utf-8")

print("Scientific evaluation report saved:")
print(report_file)


Scientific evaluation report saved:
/content/drive/MyDrive/Industrial_AI_Artifacts/phase04/phase04_evaluation_summary.md


## 29. Reproducibility Verification

In [29]:
# Verify saved evaluation artifacts and key results

saved_results = pd.read_csv(
    output_dir / "phase04_event_window_summary.csv"
)

report_file = output_dir / "phase04_evaluation_summary.md"

assert len(saved_results) == 8

assert (
    saved_results["evaluation_status"]
    .eq("not_evaluable")
    .sum() == 1
)

assert (
    saved_results.loc[
        saved_results["event_id"] == 40,
        "prediction_window_coverage_percent"
    ].iloc[0] == 50.0
)

assert report_file.exists()

print("Reproducibility checks passed.")
print("Evaluated events:", len(saved_results))
print("Scientific report exists:", report_file.exists())


Reproducibility checks passed.
Evaluated events: 8
Scientific report exists: True
